# 03 Project-specific event derivation

This notebook is optional.

Notebook 01 already performs raw BIDS conversion and trigger extraction. This notebook starts from those existing trigger-derived `events.tsv` files and derives analysis-ready event tables for the project.

Conceptual distinction:

- Trigger anchors are technical timing markers from the recording.
- Analysis events are the events you actually want to epoch or model.
- In simple projects, these can be identical.
- In this project-specific workflow, trigger anchors can mark blocks or sequences, while note-level analysis events can be derived from anchor timing plus project metadata such as `notes.csv`.

This notebook writes derivative analysis events by default:

    derivatives/meeg-pipeline/sub-*/meg/*_desc-analysis_events.tsv

It does not overwrite the raw BIDS `*_events.tsv` files from Notebook 01.

For encoding/decoding-style analyses, prefer deriving **all note-level events** and keeping variables such as `non_diatonic`, `first_non_diatonic`, `modulation_distance`, or `certainty` as metadata rather than writing only a small subset of conditions.


## Setup

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

from meeg_pipeline.bids import read_raw_bids_recording_if_exists
from meeg_pipeline.config import load_config
from meeg_pipeline.event_derivatives import make_analysis_events_path, write_analysis_events
from meeg_pipeline.workflow import (
    analysis_events_file_overview_to_dataframe,
    existing_output_policy_for_step,
    iter_recordings,
    raw_events_path,
    read_raw_events,
    recording_label,
    safe_join,
    selected_recordings_to_dataframe,
    should_overwrite,
    sorted_nonmissing_unique,
)

def find_project_root(start: Path | None = None) -> Path:
    """Find project root by searching upward for configs/local.yaml."""
    start = Path.cwd() if start is None else Path(start).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "configs" / "local.yaml").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find project root by searching for configs/local.yaml "
        f"above {start}"
    )


PROJECT_ROOT = find_project_root()
CONFIG_PATH = PROJECT_ROOT / "configs" / "local.yaml"
config = load_config(CONFIG_PATH)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CONFIG_PATH:", CONFIG_PATH)


def optional_as_set(value: Any) -> set[Any] | None:
    """Normalize a scalar/list selector to a set, preserving None as ignore."""
    if value is None:
        return None

    if isinstance(value, (list, tuple, set)):
        return set(value)

    return {value}


def select_anchor_events(
    trigger_events: pd.DataFrame,
    *,
    trial_types: list[str] | tuple[str, ...] | set[str] | str | None = None,
    values: list[int] | tuple[int, ...] | set[int] | int | None = None,
) -> pd.DataFrame:
    """Select trigger events that should act as block/sequence anchors."""
    anchors = trigger_events.copy()

    trial_type_set = optional_as_set(trial_types)
    value_set = optional_as_set(values)

    if trial_type_set is not None:
        anchors = anchors[anchors["trial_type"].isin(trial_type_set)]

    if value_set is not None:
        anchors = anchors[anchors["value"].isin(value_set)]

    return anchors.reset_index(drop=True)


def load_notes_metadata(
    notes_path: str | Path,
    *,
    events_per_block: int,
    notes_per_cycle: int,
    skip_event_ids_before: int = 0,
) -> pd.DataFrame:
    """Load project-specific note metadata and add normalized helper columns.

    The original example project logic uses global note-event IDs from the row index:

    - ``block_index = event_id // events_per_block``
    - ``note_position = event_id % events_per_block``
    - ``note_index = event_id % notes_per_cycle``

    In this project, one block contains 32 note events, while the cyclic
    musical ``note_index`` runs from 0 to 15 and therefore occurs twice per
    block. The first ``skip_event_ids_before`` rows are dropped after assigning
    global IDs, matching the old ``if id < 16: continue`` behavior.
    """
    notes_path = Path(notes_path).expanduser()

    if not notes_path.is_absolute():
        notes_path = (Path.cwd() / notes_path).resolve()

    notes = pd.read_csv(notes_path)

    column_map = {
        "note": "note_degree",
        "key signature": "key_signature",
        "scale degree": "scale_degree",
        "non-diatonic": "non_diatonic",
        "steps": "steps",
        "direction change": "direction_change",
        "jumps up the circle of fifths": "circle_of_fifths",
        "modulation distance": "modulation_distance",
        "certainty (1-7)": "certainty",
        "part of chord": "part_of_chord",
        "diatonic": "diatonic",
        "new information": "new_information",
    }

    available_column_map = {
        original: normalized
        for original, normalized in column_map.items()
        if original in notes.columns
    }

    notes = notes.rename(columns=available_column_map).copy()

    notes["event_id"] = np.arange(len(notes), dtype=int)
    notes = notes.loc[notes["event_id"] >= skip_event_ids_before].copy()

    notes["note_id"] = notes["event_id"]
    notes["block_index"] = notes["event_id"] // events_per_block
    notes["note_position"] = notes["event_id"] % events_per_block
    notes["position_in_block"] = notes["note_position"]  # backwards-compatible alias
    notes["note_index"] = notes["event_id"] % notes_per_cycle

    feature_columns = [
        "note_degree",
        "key_signature",
        "scale_degree",
        "non_diatonic",
        "steps",
        "direction_change",
        "circle_of_fifths",
        "modulation_distance",
        "certainty",
        "part_of_chord",
        "diatonic",
        "new_information",
    ]

    for column in feature_columns:
        if column not in notes.columns:
            notes[column] = pd.NA

        notes[column] = pd.to_numeric(
            notes[column],
            errors="coerce",
        ).astype("Int64")

    if notes["modulation_distance"].isna().all() and not notes["circle_of_fifths"].isna().all():
        notes["modulation_distance"] = notes["circle_of_fifths"]

    non_diatonic_position = notes["non_diatonic"].fillna(0).astype(int)
    notes["non_diatonic_position"] = non_diatonic_position.astype("Int64")
    notes["is_non_diatonic"] = (non_diatonic_position > 0).astype(int)
    notes["first_non_diatonic"] = (non_diatonic_position == 1).astype(int)
    notes["later_non_diatonic"] = (non_diatonic_position > 1).astype(int)

    helper_columns = [
        "event_id",
        "note_id",
        "note_index",
        "block_index",
        "note_position",
        "position_in_block",
    ]

    for column in helper_columns:
        notes[column] = notes[column].astype(int)

    return notes.reset_index(drop=True)


def match_note_ids(
    notes: pd.DataFrame,
    *,
    note_index: Any = None,
    note_degree: Any = None,
    key_signature: Any = None,
    scale_degree: Any = None,
    non_diatonic_id: Any = None,
    step: Any = None,
    direction_change: Any = None,
    circle_of_fifths: Any = None,
    certainty: Any = None,
    is_non_diatonic: Any = None,
    first_non_diatonic: Any = None,
    later_non_diatonic: Any = None,
    non_diatonic_position: Any = None,
    modulation_distance: Any = None,
) -> list[int]:
    """Match project-specific note-event IDs based on note metadata.

    ``notes`` is expected to have already been filtered by
    ``skip_event_ids_before`` in ``load_notes_metadata``.
    """
    matched = notes.copy()

    criteria = [
        ("note_index", note_index),
        ("note_degree", note_degree),
        ("key_signature", key_signature),
        ("scale_degree", scale_degree),
        ("non_diatonic", non_diatonic_id),
        ("steps", step),
        ("direction_change", direction_change),
        ("circle_of_fifths", circle_of_fifths),
        ("certainty", certainty),
        ("is_non_diatonic", is_non_diatonic),
        ("first_non_diatonic", first_non_diatonic),
        ("later_non_diatonic", later_non_diatonic),
        ("non_diatonic_position", non_diatonic_position),
        ("modulation_distance", modulation_distance),
    ]

    for column, value in criteria:
        value_set = optional_as_set(value)
        if value_set is not None:
            matched = matched[matched[column].isin(value_set)]

    return sorted(int(note_id) for note_id in matched["note_id"].to_list())


def selection_labels_for_note_id(
    note_id: int,
    selection_id_sets: dict[str, set[int]],
) -> list[str]:
    return [
        label
        for label, note_ids in selection_id_sets.items()
        if note_id in note_ids
    ]


def nullable_int(value: Any) -> int | pd.NA:
    """Convert a value to int if possible, otherwise return pandas NA."""
    if pd.isna(value):
        return pd.NA

    return int(value)


def derive_analysis_events_for_recording(
    *,
    recording: dict[str, str | None],
    raw,
    trigger_events: pd.DataFrame,
    notes: pd.DataFrame,
    selected_note_ids: list[int],
    selection_id_sets: dict[str, set[int]],
    anchor_trial_types: Any = None,
    anchor_values: Any = None,
    note_step_s: float = 0.25,
    event_value_offset: int = 0,
    task_block_offsets: dict[str, int] | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Derive analysis events from block anchors plus project-specific note metadata."""
    anchors = select_anchor_events(
        trigger_events,
        trial_types=anchor_trial_types,
        values=anchor_values,
    )

    task = recording.get("task")
    task_block_offset = 0

    if task_block_offsets is not None and task is not None:
        task_block_offset = int(task_block_offsets.get(str(task), 0))

    sfreq = float(raw.info["sfreq"])
    step_samples = int(round(sfreq * note_step_s))

    rows: list[dict[str, Any]] = []
    issue_rows: list[dict[str, Any]] = []

    notes_by_id = notes.set_index("note_id", drop=False)

    for note_id in sorted(selected_note_ids):
        if note_id not in notes_by_id.index:
            issue_rows.append(
                {
                    "recording": recording_label(recording),
                    "note_id": note_id,
                    "status": "missing_note_metadata",
                    "message": "Selected note_id not found in notes metadata.",
                }
            )
            continue

        note = notes_by_id.loc[note_id]
        event_id = int(note["event_id"])
        block_index = int(note["block_index"])
        note_position = int(note["note_position"])
        note_index = int(note["note_index"])
        recording_block_index = block_index - task_block_offset

        if recording_block_index < 0:
            continue

        if recording_block_index >= len(anchors):
            issue_rows.append(
                {
                    "recording": recording_label(recording),
                    "note_id": note_id,
                    "event_id": event_id,
                    "status": "missing_anchor",
                    "message": (
                        f"block_index={block_index} maps to recording_block_index="
                        f"{recording_block_index}, which is out of range for "
                        f"{len(anchors)} anchors."
                    ),
                }
            )
            continue

        anchor = anchors.iloc[recording_block_index]
        anchor_sample = int(anchor["sample"])
        sample = anchor_sample + note_position * step_samples

        if not (raw.first_samp <= sample <= raw.last_samp):
            issue_rows.append(
                {
                    "recording": recording_label(recording),
                    "note_id": note_id,
                    "event_id": event_id,
                    "status": "out_of_bounds",
                    "message": (
                        f"sample={sample} outside raw bounds "
                        f"({raw.first_samp}, {raw.last_samp})."
                    ),
                }
            )
            continue

        labels = selection_labels_for_note_id(note_id, selection_id_sets)
        trial_type = "+".join(labels) if labels else "note"
        onset = (sample - raw.first_samp) / sfreq

        rows.append(
            {
                "onset": onset,
                "duration": 0.0,
                "trial_type": trial_type,
                "value": int(event_id + event_value_offset),
                "sample": int(sample),
                "anchor_sample": int(anchor_sample),
                "anchor_index": int(recording_block_index),
                "anchor_trial_type": str(anchor.get("trial_type", "")),
                "anchor_value": int(anchor["value"]) if "value" in anchor else np.nan,
                "event_id": int(event_id),
                "note_id": int(note_id),
                "note_index": int(note_index),
                "block_index": int(block_index),
                "task_block_offset": int(task_block_offset),
                "recording_block_index": int(recording_block_index),
                "note_position": int(note_position),
                "position_in_block": int(note_position),
                "note_degree": nullable_int(note["note_degree"]),
                "key_signature": nullable_int(note["key_signature"]),
                "scale_degree": nullable_int(note["scale_degree"]),
                "non_diatonic": nullable_int(note["non_diatonic"]),
                "steps": nullable_int(note["steps"]),
                "direction_change": nullable_int(note["direction_change"]),
                "circle_of_fifths": nullable_int(note["circle_of_fifths"]),
                "modulation_distance": nullable_int(note["modulation_distance"]),
                "certainty": nullable_int(note["certainty"]),
                "non_diatonic_position": nullable_int(note["non_diatonic_position"]),
                "is_non_diatonic": nullable_int(note["is_non_diatonic"]),
                "first_non_diatonic": nullable_int(note["first_non_diatonic"]),
                "later_non_diatonic": nullable_int(note["later_non_diatonic"]),
                "part_of_chord": nullable_int(note["part_of_chord"]),
                "diatonic": nullable_int(note["diatonic"]),
                "new_information": nullable_int(note["new_information"]),
                "selection_labels": safe_join(labels),
            }
        )

    events = pd.DataFrame(rows).sort_values("onset").reset_index(drop=True)
    issues = pd.DataFrame(issue_rows)

    return events, issues


## Selection

Use the same selection style as the other notebooks.


In [ ]:
SUBJECTS = "all"
SESSIONS = "all"
TASKS = "all"
RUNS = "all"

# Single-file/manual inspection cells at the end of notebooks are disabled by default
# so batch runs over many participants do not stop for plots or ad-hoc file views.
RUN_SINGLE_FILE_INSPECTIONS = False
selected_recordings = list(
    iter_recordings(
        config,
        subjects=SUBJECTS,
        sessions=SESSIONS,
        tasks=TASKS,
        runs=RUNS,
    )
)

selected_recordings_to_dataframe(selected_recordings)


## Overwrite policy

This notebook writes derivative analysis-event TSV files.

Default:

    OVERWRITE_STEPS = []

This skips existing analysis-event files. To recompute them:

    OVERWRITE_STEPS = ["analysis_events"]


In [ ]:
OVERWRITE_STEPS = []

pd.DataFrame(
    [
        {
            "step": "analysis_events",
            "overwrite": should_overwrite("analysis_events", OVERWRITE_STEPS),
            "policy": existing_output_policy_for_step(
                "analysis_events",
                OVERWRITE_STEPS,
            ),
        }
    ]
)


## Note-level event-derivation settings

Adjust these settings for the concrete project. The defaults below match the tonal-key-change example, but the derivation pattern is generic: trigger anchors define blocks, while metadata rows define note-level analysis events.

Important:

- `EVENTS_PER_BLOCK = 32` means that one MEG trigger/anchor represents one block with 32 derived note-level events.
- `NOTES_PER_CYCLE = 16` means that the musical `note_index` is cyclic and runs from 0 to 15. Therefore each `note_index` occurs twice in one 32-event block.
- `NOTE_STEP_S` controls the time between derived note positions within a block.
- Your old master thesis code used `NOTE_STEP_S = 0.25`.
- `SKIP_EVENT_IDS_BEFORE = 16` reproduces the old `if id < 16: continue` rule.

`ANCHOR_VALUES` and `ANCHOR_TRIAL_TYPES` can restrict which trigger rows from the raw BIDS events.tsv are used as block anchors. Keep both `None` to use all trigger-derived events as anchors.

`TASK_BLOCK_OFFSETS` maps task names to the corresponding block offset in the global `notes.csv`. This is needed when one metadata table contains multiple task-specific sequences.


In [ ]:
NOTES_PATH = PROJECT_ROOT / "stimuli" / "notes.csv"

print(NOTES_PATH)
print(NOTES_PATH.exists())

EVENTS_PER_BLOCK = 32
NOTES_PER_CYCLE = 16
NOTE_STEP_S = 0.25
SKIP_EVENT_IDS_BEFORE = 16
EVENT_VALUE_OFFSET = 0

ANCHOR_TRIAL_TYPES = None
ANCHOR_VALUES = None

# Tonalkey-specific: map each task to its starting block in the global notes.csv.
# Leave empty if each recording starts at block 0 in notes.csv.
TASK_BLOCK_OFFSETS: dict[str, int] = {}

ANALYSIS_EVENTS_DESC = "analysis"
DERIVE_ALL_NOTE_EVENTS = True

settings_table = pd.DataFrame(
    [
        {
            "notes_path": str(NOTES_PATH),
            "events_per_block": EVENTS_PER_BLOCK,
            "notes_per_cycle": NOTES_PER_CYCLE,
            "note_step_s": NOTE_STEP_S,
            "skip_event_ids_before": SKIP_EVENT_IDS_BEFORE,
            "event_value_offset": EVENT_VALUE_OFFSET,
            "anchor_trial_types": ANCHOR_TRIAL_TYPES,
            "anchor_values": ANCHOR_VALUES,
            "task_block_offsets": TASK_BLOCK_OFFSETS,
            "analysis_events_desc": ANALYSIS_EVENTS_DESC,
            "derive_all_note_events": DERIVE_ALL_NOTE_EVENTS,
        }
    ]
)

settings_table


## Load note metadata

The expected metadata columns follow the tonal-key-change example note-metadata table:

- `note`
- `key signature`
- `scale degree`
- `non-diatonic`
- `steps`
- `direction change`
- `jumps up the circle of fifths`
- `certainty (1-7)`

The notebook adds normalized helper columns such as `note_id`, `note_index`, `block_index`, and `position_in_block`.

For encoding analyses, the important point is that every note can be retained as one event while tonal-key-change variables remain available as metadata columns. Derived convenience columns include `is_non_diatonic`, `first_non_diatonic`, `later_non_diatonic`, `non_diatonic_position`, and `modulation_distance`.


In [ ]:
notes = load_notes_metadata(
    NOTES_PATH,
    events_per_block=EVENTS_PER_BLOCK,
    notes_per_cycle=NOTES_PER_CYCLE,
    skip_event_ids_before=SKIP_EVENT_IDS_BEFORE,
)

notes.head(20)


## Note metadata summary

In [ ]:
pd.DataFrame(
    [
        {
            "n_note_events": len(notes),
            "n_blocks": int(notes["block_index"].max()) + 1,
            "events_per_block": EVENTS_PER_BLOCK,
            "notes_per_cycle": NOTES_PER_CYCLE,
            "min_event_id": int(notes["event_id"].min()),
            "max_event_id": int(notes["event_id"].max()),
            "note_indices": safe_join(
                sorted_nonmissing_unique(notes["note_index"])
            ),
            "note_positions": safe_join(
                [
                    int(notes["note_position"].min()),
                    int(notes["note_position"].max()),
                ]
            ),
            "key_signatures": safe_join(
                sorted_nonmissing_unique(notes["key_signature"])
            ),
            "non_diatonic_values": safe_join(
                sorted_nonmissing_unique(notes["non_diatonic"])
            ),
            "steps": safe_join(
                sorted_nonmissing_unique(notes["steps"])
            ),
            "direction_changes": safe_join(
                sorted_nonmissing_unique(notes["direction_change"])
            ),
            "circle_of_fifths": safe_join(
                sorted_nonmissing_unique(notes["circle_of_fifths"])
            ),
            "modulation_distance": safe_join(
                sorted_nonmissing_unique(notes["modulation_distance"])
            ),
            "first_non_diatonic_count": int(notes["first_non_diatonic"].sum()),
            "non_diatonic_count": int(notes["is_non_diatonic"].sum()),
        }
    ]
)


## Define analysis-event selections

For encoding-style analyses, derive all note-level events by default and keep the condition-defining variables as metadata. This preserves the diatonic reference notes and avoids throwing away events that may be useful as covariates or controls.

Set `DERIVE_ALL_NOTE_EVENTS = False` if you only want to write the union of named selections.

The example named selections below are still useful because they label selected events through `trial_type`/`selection_labels` while all unlabeled note events remain available as `trial_type == "note"`.


In [ ]:
EVENT_SELECTIONS = {
    "first_non_diatonic": {
        "first_non_diatonic": 1,
    },
    "non_diatonic": {
        "is_non_diatonic": 1,
    },
    "key_change": {
        "note_index": 0,
    },
}

selection_id_sets = {
    label: set(
        match_note_ids(
            notes,
            **criteria,
        )
    )
    for label, criteria in EVENT_SELECTIONS.items()
}

if DERIVE_ALL_NOTE_EVENTS:
    selected_note_ids = sorted(int(note_id) for note_id in notes["note_id"].to_list())
else:
    selected_note_ids = sorted(set().union(*selection_id_sets.values()))

selection_summary_rows = []

selection_summary_rows.append(
    {
        "selection": "all_note_events" if DERIVE_ALL_NOTE_EVENTS else "selected_union_only",
        "n_note_ids": len(selected_note_ids),
        "first_note_ids": safe_join(selected_note_ids[:10]),
        "last_note_ids": safe_join(selected_note_ids[-10:]),
    }
)

for label, note_ids in selection_id_sets.items():
    selection_summary_rows.append(
        {
            "selection": label,
            "n_note_ids": len(note_ids),
            "first_note_ids": safe_join(sorted(note_ids)[:10]),
            "last_note_ids": safe_join(sorted(note_ids)[-10:]),
        }
    )

selection_summary = pd.DataFrame(selection_summary_rows)
selection_summary

## Preview selected note metadata

In [ ]:
notes[notes["note_id"].isin(selected_note_ids)].head(30)


## Trigger-anchor overview

This reads the existing raw BIDS events.tsv files written by Notebook 01 and selects anchor rows according to `ANCHOR_TRIAL_TYPES` / `ANCHOR_VALUES`.


In [ ]:
anchor_overview_rows = []

for recording in selected_recordings:
    trigger_events = read_raw_events(config, recording)
    raw_result = read_raw_bids_recording_if_exists(
        config,
        subject=recording["subject"],
        session=recording["session"],
        task=recording["task"],
        run=recording["run"],
        preload=False,
    )

    if trigger_events is None:
        anchor_overview_rows.append(
            {
                "recording": recording_label(recording),
                "status": "missing_events_tsv",
                "n_trigger_events": 0,
                "n_anchors": 0,
                "raw_status": raw_result.status,
                "message": "Raw BIDS events.tsv does not exist.",
            }
        )
        continue

    anchors = select_anchor_events(
        trigger_events,
        trial_types=ANCHOR_TRIAL_TYPES,
        values=ANCHOR_VALUES,
    )

    anchor_overview_rows.append(
        {
            "recording": recording_label(recording),
            "status": "loaded",
            "n_trigger_events": len(trigger_events),
            "n_anchors": len(anchors),
            "expected_derived_note_events": len(anchors) * EVENTS_PER_BLOCK,
            "raw_status": raw_result.status,
            "first_anchor_sample": int(anchors["sample"].iloc[0]) if len(anchors) else None,
            "last_anchor_sample": int(anchors["sample"].iloc[-1]) if len(anchors) else None,
            "trial_types": safe_join(sorted(trigger_events["trial_type"].unique()))
            if "trial_type" in trigger_events
            else "",
            "values": safe_join(sorted(trigger_events["value"].unique()))
            if "value" in trigger_events
            else "",
            "message": "",
        }
    )

anchor_overview = pd.DataFrame(anchor_overview_rows)
anchor_overview


## Task-specific block mapping check

This diagnostic checks whether the global notes metadata table is being mapped to the task-specific trigger anchors with the intended block offsets.


In [ ]:
mapping_rows = []

for recording in selected_recordings:
    trigger_events = read_raw_events(config, recording)
    if trigger_events is None:
        n_anchors = 0
    else:
        n_anchors = len(
            select_anchor_events(
                trigger_events,
                trial_types=ANCHOR_TRIAL_TYPES,
                values=ANCHOR_VALUES,
            )
        )

    task = recording.get("task")
    offset = int(TASK_BLOCK_OFFSETS.get(str(task), 0)) if task is not None else 0

    first_global_block = offset
    last_global_block = offset + n_anchors - 1 if n_anchors else None

    matching_notes = notes[
        (notes["block_index"] >= first_global_block)
        & (notes["block_index"] <= last_global_block)
    ] if n_anchors else notes.iloc[0:0]

    mapping_rows.append(
        {
            "recording": recording_label(recording),
            "task": task,
            "task_block_offset": offset,
            "n_anchors": n_anchors,
            "first_global_block": first_global_block if n_anchors else None,
            "last_global_block": last_global_block,
            "n_matching_note_events": len(matching_notes),
            "expected_note_events_from_anchors": n_anchors * EVENTS_PER_BLOCK,
            "events_per_block": EVENTS_PER_BLOCK,
            "notes_per_cycle": NOTES_PER_CYCLE,
        }
    )

task_block_mapping_check = pd.DataFrame(mapping_rows)
task_block_mapping_check


## Derive analysis events

This combines:

- raw timing information
- existing trigger anchors from Notebook 01
- note metadata from `notes.csv`
- the selected note IDs from the project-specific matching logic

Outputs are derivative `*_desc-analysis_events.tsv` files.


In [ ]:
derive_rows = []
issue_tables = []
preview_tables = []

analysis_events_policy = existing_output_policy_for_step(
    "analysis_events",
    OVERWRITE_STEPS,
)

for recording in selected_recordings:
    label = recording_label(recording)
    trigger_events = read_raw_events(config, recording)
    trigger_events_path = raw_events_path(config, recording)

    raw_result = read_raw_bids_recording_if_exists(
        config,
        subject=recording["subject"],
        session=recording["session"],
        task=recording["task"],
        run=recording["run"],
        preload=False,
    )

    output_path = make_analysis_events_path(
        config,
        subject=recording["subject"],
        session=recording["session"],
        task=recording["task"],
        run=recording["run"],
        desc=ANALYSIS_EVENTS_DESC,
    )

    if trigger_events is None:
        derive_rows.append(
            {
                "recording": label,
                "status": "missing_events_tsv",
                "n_events": 0,
                "n_issues": None,
                "path": str(output_path),
                "message": "Raw BIDS events.tsv does not exist.",
            }
        )
        continue

    if raw_result.raw is None:
        derive_rows.append(
            {
                "recording": label,
                "status": raw_result.status,
                "n_events": 0,
                "n_issues": None,
                "path": str(output_path),
                "message": raw_result.message,
            }
        )
        continue

    analysis_events, issues = derive_analysis_events_for_recording(
        recording=recording,
        raw=raw_result.raw,
        trigger_events=trigger_events,
        notes=notes,
        selected_note_ids=selected_note_ids,
        selection_id_sets=selection_id_sets,
        anchor_trial_types=ANCHOR_TRIAL_TYPES,
        anchor_values=ANCHOR_VALUES,
        note_step_s=NOTE_STEP_S,
        event_value_offset=EVENT_VALUE_OFFSET,
        task_block_offsets=TASK_BLOCK_OFFSETS,
    )

    write_result = write_analysis_events(
        config,
        analysis_events,
        subject=recording["subject"],
        session=recording["session"],
        task=recording["task"],
        run=recording["run"],
        desc=ANALYSIS_EVENTS_DESC,
        on_existing=analysis_events_policy,
        sidecar_description=(
            "Tonalkey-specific note-level analysis events derived from "
            "trigger anchors and notes.csv metadata. One anchor represents "
            f"one block with {EVENTS_PER_BLOCK} note events; note_index is "
            f"cyclic over {NOTES_PER_CYCLE} positions."
        ),
        source_events=str(trigger_events_path),
    )

    derive_rows.append(
        {
            "recording": label,
            "status": write_result.status,
            "n_events": len(analysis_events),
            "n_issues": len(issues),
            "path": write_result.events_path,
            "sidecar_path": write_result.sidecar_path,
            "message": write_result.message,
        }
    )

    if len(issues):
        issue_tables.append(issues)

    if len(analysis_events):
        preview = analysis_events.head(10).copy()
        preview.insert(0, "recording", label)
        preview_tables.append(preview)

derive_results = pd.DataFrame(derive_rows)
derive_results


## Derivation issues

In [ ]:
if issue_tables:
    derivation_issues = pd.concat(issue_tables, ignore_index=True)
else:
    derivation_issues = pd.DataFrame(
        columns=["recording", "note_id", "event_id", "status", "message"]
    )

derivation_issues


## Preview derived analysis events

In [ ]:
if preview_tables:
    derived_events_preview = pd.concat(preview_tables, ignore_index=True)
else:
    derived_events_preview = pd.DataFrame()

derived_events_preview


## Analysis-event file overview

Run this to inspect the existing derivative analysis-event files.


In [ ]:
analysis_file_overview = analysis_events_file_overview_to_dataframe(
    config,
    selected_recordings,
    desc=ANALYSIS_EVENTS_DESC,
)

analysis_file_overview


## Inspect one derived analysis-events file

Use this for detailed checking of one output table.

> Disabled by default for batch runs. Set `RUN_SINGLE_FILE_INSPECTIONS = True` in the selection cell to run this section.



In [ ]:
if RUN_SINGLE_FILE_INSPECTIONS:
    INSPECT_SUBJECT = "0001"
    INSPECT_SESSION = None
    INSPECT_TASK = "example"
    INSPECT_RUN = None

    inspect_recording = {
        "subject": INSPECT_SUBJECT.removeprefix("sub-"),
        "session": INSPECT_SESSION,
        "task": INSPECT_TASK,
        "run": INSPECT_RUN,
    }

    inspect_path = make_analysis_events_path(
        config,
        subject=inspect_recording["subject"],
        session=inspect_recording["session"],
        task=inspect_recording["task"],
        run=inspect_recording["run"],
        desc=ANALYSIS_EVENTS_DESC,
    )

    if inspect_path.exists():
        inspect_events = pd.read_csv(inspect_path, sep="\t")
    else:
        inspect_events = pd.DataFrame()

    pd.DataFrame(
        [
            {
                "recording": recording_label(inspect_recording),
                "exists": inspect_path.exists(),
                "n_events": len(inspect_events),
                "path": str(inspect_path),
            }
        ]
    )
else:
    print('Skipped single-file inspection cell 30 in 1B_preprocessing/03_project_specific_events.ipynb. Set RUN_SINGLE_FILE_INSPECTIONS = True to run it.')


In [ ]:
if RUN_SINGLE_FILE_INSPECTIONS:
    inspect_events.head(30)
else:
    print('Skipped single-file inspection cell 31 in 1B_preprocessing/03_project_specific_events.ipynb. Set RUN_SINGLE_FILE_INSPECTIONS = True to run it.')


In [ ]:
if RUN_SINGLE_FILE_INSPECTIONS:
    if inspect_events.empty:
        pd.DataFrame()
    else:
        inspect_events.groupby("trial_type").size().reset_index(name="n_events")
else:
    print('Skipped single-file inspection cell 32 in 1B_preprocessing/03_project_specific_events.ipynb. Set RUN_SINGLE_FILE_INSPECTIONS = True to run it.')
